In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import io
import random 
import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('HR-Employee-Attrition.csv')

In [3]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
df = df.drop(['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours'], axis=1)

In [5]:
# Here we encode categorical variables using Label Encoding
# This is a simple approach; for more complex datasets, consider using One-Hot Encoding or other techniques.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col]) 

In [6]:
target = df['Attrition']
X = df.drop('Attrition', axis=1)

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.2, random_state=7, stratify=target)

In [9]:
target.value_counts(normalize=True)

Attrition
0    0.838776
1    0.161224
Name: proportion, dtype: float64

In [10]:
from sklearn.tree import DecisionTreeClassifier

In [11]:
tree_1 = DecisionTreeClassifier(random_state=1)

In [12]:
tree_1.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [13]:
from sklearn.metrics import confusion_matrix, classification_report

In [14]:
y_pred = tree_1.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[215  32]
 [ 25  22]]
              precision    recall  f1-score   support

           0       0.90      0.87      0.88       247
           1       0.41      0.47      0.44        47

    accuracy                           0.81       294
   macro avg       0.65      0.67      0.66       294
weighted avg       0.82      0.81      0.81       294



In [15]:
y_train_pred = tree_1.predict(X_train)
print(confusion_matrix(y_train, y_train_pred))  
print(classification_report(y_train, y_train_pred))

[[986   0]
 [  0 190]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       986
           1       1.00      1.00      1.00       190

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176



### Here we can see the classification report of traing data and test data. After this we can say that our model is <b>Overfitted</b> 

In [16]:
from sklearn.metrics import accuracy_score, classification_report

def classification_performance(model, features, target, dataset_name_string):
    print(f"Classification Performance for {dataset_name_string} Dataset:")
    print()

    predicted_target = model.predict(features)
    report = pd.DataFrame(classification_report(target, predicted_target, output_dict=True))
    print(report)
    print()
    print(f"Accuracy: {accuracy_score(target, predicted_target)*100}")
    
    

In [17]:
from sklearn.model_selection import KFold, cross_val_score

def kfold_cross_validation_score(model, features, target):
    kfold = KFold(n_splits=10)
    result = cross_val_score(model, features, target, cv=kfold, scoring='accuracy')

    print(f"Cross-Validation Accuracy Scores: {result}")
    print(f"Mean Cross-Validation Accuracy: {result.mean()*100:.2f}%")
    print(f"Standard Deviation of Cross-Validation Accuracy: {result.std()*100:.2f}%")
   

In [18]:
classification_performance(tree_1, X_train, y_train, "Training")

Classification Performance for Training Dataset:

               0      1  accuracy  macro avg  weighted avg
precision    1.0    1.0       1.0        1.0           1.0
recall       1.0    1.0       1.0        1.0           1.0
f1-score     1.0    1.0       1.0        1.0           1.0
support    986.0  190.0       1.0     1176.0        1176.0

Accuracy: 100.0


In [19]:
classification_performance(tree_1, X_test, y_test, "Testing")

Classification Performance for Testing Dataset:

                    0          1  accuracy   macro avg  weighted avg
precision    0.895833   0.407407  0.806122    0.651620      0.817752
recall       0.870445   0.468085  0.806122    0.669265      0.806122
f1-score     0.882957   0.435644  0.806122    0.659300      0.811448
support    247.000000  47.000000  0.806122  294.000000    294.000000

Accuracy: 80.61224489795919


In [20]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

In [23]:
bag_classifier = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=1), n_estimators=10, random_state=7, oob_score=True)

In [24]:
bag_model = bag_classifier.fit(X_train, y_train)

In [25]:
oob_score = bag_model.oob_score_
print(f"OOB Score: {oob_score*100:.2f}%")

OOB Score: 82.40%


In [26]:
classification_performance(bag_model, X_train, y_train, "Training")

Classification Performance for Training Dataset:

                    0           1  accuracy    macro avg  weighted avg
precision    0.980119    1.000000  0.982993     0.990060      0.983331
recall       1.000000    0.894737  0.982993     0.947368      0.982993
f1-score     0.989960    0.944444  0.982993     0.967202      0.982606
support    986.000000  190.000000  0.982993  1176.000000   1176.000000

Accuracy: 98.29931972789116


In [27]:
classification_performance(bag_model, X_test, y_test, "Testing")

Classification Performance for Testing Dataset:

                    0          1  accuracy   macro avg  weighted avg
precision    0.874539   0.565217   0.85034    0.719878      0.825089
recall       0.959514   0.276596   0.85034    0.618055      0.850340
f1-score     0.915058   0.371429   0.85034    0.643243      0.828151
support    247.000000  47.000000   0.85034  294.000000    294.000000

Accuracy: 85.03401360544217


When Accuracy score differ more then 10 percent from training to testing then it's an overfitting model.

Morely if we check the f1 score then testing score is make a huge difference then training model outcome.

In [28]:
kfold_cross_validation_score(bag_model, X, target)

Cross-Validation Accuracy Scores: [0.85034014 0.85034014 0.86394558 0.84353741 0.82993197 0.82993197
 0.82312925 0.85034014 0.85034014 0.8707483 ]
Mean Cross-Validation Accuracy: 84.63%
Standard Deviation of Cross-Validation Accuracy: 1.43%


In [29]:
rf_classifier = RandomForestClassifier(
    n_estimators=25,
    max_depth=10,# We can generalize the model by limiting the depth of the trees
    min_impurity_decrease=0.05,# We can set a minimum impurity decrease to prevent the model from creating very specific rules that only apply to the training data
    max_samples=1.0,# we are using all originl samples to bootstrap the trees
    oob_score=True,
    class_weight='balanced',# we can assign higher weights to the minority class to help the model learn from it better
    random_state=7
)

In [30]:
rf_model = rf_classifier.fit(X_train, y_train)

In [31]:
classification_performance(rf_model, X_train, y_train, "Training")

Classification Performance for Training Dataset:

                    0           1  accuracy    macro avg  weighted avg
precision    0.909091    0.320442  0.727891     0.614766      0.813986
recall       0.750507    0.610526  0.727891     0.680517      0.727891
f1-score     0.822222    0.420290  0.727891     0.621256      0.757284
support    986.000000  190.000000  0.727891  1176.000000   1176.000000

Accuracy: 72.78911564625851


In [32]:
classification_performance(rf_model, X_test, y_test, "Testing")

Classification Performance for Testing Dataset:

                    0          1  accuracy   macro avg  weighted avg
precision    0.888372   0.291139  0.727891    0.589756      0.792896
recall       0.773279   0.489362  0.727891    0.631321      0.727891
f1-score     0.826840   0.365079  0.727891    0.595960      0.753021
support    247.000000  47.000000  0.727891  294.000000    294.000000

Accuracy: 72.78911564625851
